# AADNet FDR-XAI: Channel Importance Analysis

Applies **False Discovery Rate (FDR)** correction to **Leave-One-Channel-Out (LOCO)**
accuracy data from trained AADNet models on the DTU EEG dataset.

### Approach
1. **Channel importance** = SS baseline accuracy − LOCO accuracy (drop when channel is ablated)
2. **Bootstrap 95% CIs** across 18 subjects
3. **Sign-flip permutation test** → p-value per channel (H₀: importance = 0)
4. **Benjamini-Hochberg FDR** correction at α = 0.05
5. **ROI-level** pooling and FDR

### Required files
| File | Description |
|------|-------------|
| `results_gce/results/channel_distribute_AADNet_DTU_LOCO_acc.npy` | LOCO test accuracies `(1, 6, 18, 64)` |
| `results_gce/results/SS_AADNet_DTU_final_SS_acc.npy` | SS baseline accuracies `(1, 6, 18)` |

In [ ]:
import sys
import json
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt

%matplotlib inline

ROOT        = Path('..').resolve()
RESULTS_DIR = ROOT / 'results_gce' / 'results'
OUTPUT_DIR  = ROOT / 'results_gce'

# ── Statistical parameters ───────────────────────────────────────────────────
N_BOOT    = 1000
N_PERM    = 5000
FDR_ALPHA = 0.05
SEED      = 42
WINDOWS   = [1, 2, 5, 10, 20, 40]   # seconds

print(f'ROOT        : {ROOT}')
print(f'RESULTS_DIR : {RESULTS_DIR}')
print(f'Exists      : {RESULTS_DIR.exists()}')

## Section A — Load LOCO & Baseline Data

In [ ]:
loco_raw = np.load(RESULTS_DIR / 'channel_distribute_AADNet_DTU_LOCO_acc.npy')
base_raw = np.load(RESULTS_DIR / 'SS_AADNet_DTU_final_SS_acc.npy')

print(f'LOCO shape     : {loco_raw.shape}   expected (n_configs, n_windows, n_subjects, n_channels)')
print(f'Baseline shape : {base_raw.shape}   expected (n_reps, n_windows, n_subjects)')

# Squeeze the leading singleton dim
loco = loco_raw[0]   # (6, 18, 64)
base = base_raw[0]   # (6, 18)

N_WINDOWS, N_SUBJECTS, N_CHANNELS = loco.shape
print(f'\n  Windows  : {N_WINDOWS}  {WINDOWS}')
print(f'  Subjects : {N_SUBJECTS}')
print(f'  Channels : {N_CHANNELS}')
print(f'\nMean SS baseline accuracy (all windows): {base.mean():.3f}')
print(f'Mean LOCO accuracy         (all windows): {loco.mean():.3f}')

In [ ]:
# DTU 64-channel names (Fuglsang montage, order from config_AADNet_SS_DTU.yml)
CHANNEL_NAMES = [
    'Fp1','AF7','AF3','F1','F3','F5','F7','FT7','FC5','FC3','FC1',
    'C1','C3','C5','T7','TP7','CP5','CP3','CP1','P1','P3','P5','P7',
    'P9','PO7','PO3','O1','Iz','Oz','POz','Pz','CPz','Fpz','Fp2',
    'AF8','AF4','AFz','Fz','F2','F4','F6','F8','FT8','FC6','FC4',
    'FC2','FCz','Cz','C2','C4','C6','T8','TP8','CP6','CP4','CP2',
    'P2','P4','P6','P8','P10','PO8','PO4','O2'
]
assert len(CHANNEL_NAMES) == 64

# ROI grouping by electrode name
ROI_MAP = {
    'Frontal':        ['Fp1','AF7','AF3','F1','F3','F5','F7','Fpz','Fp2',
                       'AF8','AF4','AFz','Fz','F2','F4','F6','F8'],
    'Fronto-Central': ['FT7','FC5','FC3','FC1','FT8','FC6','FC4','FC2','FCz'],
    'Central':        ['C1','C3','C5','Cz','C2','C4','C6'],
    'Temporal':       ['T7','TP7','T8','TP8'],
    'Parietal':       ['CP5','CP3','CP1','P1','P3','P5','P7','P9','Pz',
                       'CPz','CP6','CP4','CP2','P2','P4','P6','P8','P10'],
    'Occipital':      ['PO7','PO3','O1','Iz','Oz','POz','PO8','PO4','O2'],
}

all_roi_chs = [ch for chs in ROI_MAP.values() for ch in chs]
assert sorted(all_roi_chs) == sorted(CHANNEL_NAMES), 'ROI map does not cover all 64 channels'

CH_TO_ROI = {ch: roi for roi, chs in ROI_MAP.items() for ch in chs}

for roi, chs in ROI_MAP.items():
    print(f'{roi:>16} : {len(chs):2d} channels')
print(f'{"TOTAL":>16} : {sum(len(v) for v in ROI_MAP.values())} channels')

## Section B — Channel Importance (LOCO Delta)

**Importance** = SS baseline accuracy − LOCO accuracy.

A positive value means the channel contributes to auditory attention decoding:
ablating it degrades performance.

In [ ]:
# importance[w, s, c] = baseline_acc[w, s] - loco_acc[w, s, c]
importance = base[:, :, np.newaxis] - loco   # (6, 18, 64)

print(f'Importance shape : {importance.shape}  (windows x subjects x channels)')
print(f'Overall mean     : {importance.mean():.5f}')
print(f'Overall std      : {importance.std():.5f}')
print(f'Positive ratio   : {(importance > 0).mean():.1%}')

# Average over windows for a single ranking
imp_avg = importance.mean(axis=0)         # (18, 64) subjects x channels
imp_ch  = imp_avg.mean(axis=0)            # (64,)    mean across subjects

top10 = np.argsort(imp_ch)[::-1][:10]
print('\nTop 10 channels (avg across windows & subjects):')
print(f'{"Rank":>4}  {"Channel":>8}  {"ROI":>16}  {"Importance":>12}')
print('-' * 48)
for rank, idx in enumerate(top10, 1):
    ch = CHANNEL_NAMES[idx]
    print(f'{rank:>4}  {ch:>8}  {CH_TO_ROI[ch]:>16}  {imp_ch[idx]:>12.5f}')

## Section C — Statistical Testing

For each channel we test H₀: *mean importance = 0* across subjects using a
**sign-flip permutation test** (two-sided, 5 000 permutations). Bootstrap
95% CIs are also computed for visual reference.

In [ ]:
def bootstrap_ci(arr, n_boot=1000, ci=0.95, seed=42):
    """Bootstrap CI for the mean of arr (n_subjects,)."""
    rng = np.random.RandomState(seed)
    means = np.empty(n_boot)
    for b in range(n_boot):
        idx = rng.randint(0, len(arr), size=len(arr))
        means[b] = arr[idx].mean()
    alpha = (1 - ci) / 2
    lo, hi = np.percentile(means, [alpha * 100, (1 - alpha) * 100])
    return float(arr.mean()), float(lo), float(hi)


def sign_flip_pvalue(arr, n_perm=5000, seed=42):
    """Two-sided sign-flip permutation p-value (H0: mean = 0)."""
    rng = np.random.RandomState(seed)
    obs = abs(arr.mean())
    null = np.empty(n_perm)
    for i in range(n_perm):
        signs = rng.choice([-1, 1], size=len(arr))
        null[i] = abs((arr * signs).mean())
    return float((np.sum(null >= obs) + 1) / (n_perm + 1))


def fdr_bh(p_values, alpha=0.05):
    """Benjamini-Hochberg FDR. Returns (adjusted_p, significant_mask)."""
    n = len(p_values)
    sorted_idx = np.argsort(p_values)
    sorted_p   = p_values[sorted_idx]
    adjusted   = np.empty(n)
    adjusted[sorted_idx[-1]] = sorted_p[-1]
    for i in range(n - 2, -1, -1):
        bh_val = sorted_p[i] * n / (i + 1)
        adjusted[sorted_idx[i]] = min(bh_val, adjusted[sorted_idx[i + 1]])
    adjusted = np.clip(adjusted, 0.0, 1.0)
    return adjusted, adjusted < alpha

print('Utility functions defined.')

In [ ]:
# Use window-averaged importance for per-channel stats
# imp_avg shape: (18, 64)  — subjects x channels

print('Computing bootstrap CIs ...')
ci_arr = np.array([
    bootstrap_ci(imp_avg[:, c], n_boot=N_BOOT, seed=SEED)
    for c in range(N_CHANNELS)
])  # (64, 3): [mean, ci_lo, ci_hi]

print('Computing sign-flip p-values ...')
p_values = np.array([
    sign_flip_pvalue(imp_avg[:, c], n_perm=N_PERM, seed=SEED)
    for c in range(N_CHANNELS)
])

print(f'Done.  p-value range: [{p_values.min():.4f}, {p_values.max():.4f}]')
print(f'Channels p < 0.05 (uncorrected): {(p_values < 0.05).sum()}')
print(f'CI entirely above 0             : {(ci_arr[:, 1] > 0).sum()}')

## Section D — FDR Correction (Benjamini-Hochberg)

Applying BH-FDR at α = 0.05 to control the expected proportion of false discoveries
among the 64 simultaneous channel tests.

In [ ]:
adj_p, sig_mask = fdr_bh(p_values, alpha=FDR_ALPHA)

print(f'FDR correction (BH, alpha={FDR_ALPHA})')
print(f'  Channels tested      : {N_CHANNELS}')
print(f'  Significant channels : {sig_mask.sum()}')
print(f'  Non-significant      : {(~sig_mask).sum()}')

## Section E — Significant Channels Table

In [ ]:
sig_indices = np.where(sig_mask)[0]
sig_sorted  = sig_indices[np.argsort(adj_p[sig_indices])]

if len(sig_sorted) == 0:
    print(f'No channels survive FDR correction at alpha={FDR_ALPHA}.')
    print('Consider relaxing FDR threshold or increasing N_PERM.')
else:
    header = f'{"Ch":>4}  {"Name":>8}  {"ROI":>16}  {"Importance":>12}  {"CI_lo":>8}  {"CI_hi":>8}  {"p_raw":>8}  {"q_adj":>8}'
    print(f'Significant channels after FDR correction (q < {FDR_ALPHA}):')
    print(header)
    print('-' * len(header))
    for idx in sig_sorted:
        ch = CHANNEL_NAMES[idx]
        m, lo, hi = ci_arr[idx]
        print(f'{idx:>4}  {ch:>8}  {CH_TO_ROI[ch]:>16}  '
              f'{m:>12.5f}  {lo:>8.5f}  {hi:>8.5f}  '
              f'{p_values[idx]:>8.4f}  {adj_p[idx]:>8.4f}')

## Section F — ROI-Level Analysis

Channels within each anatomical ROI are pooled (mean importance across channels in the
ROI), then FDR is applied to the six ROI p-values.

In [ ]:
roi_names   = list(ROI_MAP.keys())
roi_results = {}

for roi_name, roi_chs in ROI_MAP.items():
    roi_indices = [CHANNEL_NAMES.index(ch) for ch in roi_chs]
    roi_imp_subj = imp_avg[:, roi_indices].mean(axis=1)   # (18,)
    mean, lo, hi = bootstrap_ci(roi_imp_subj, n_boot=N_BOOT, seed=SEED)
    p_val        = sign_flip_pvalue(roi_imp_subj, n_perm=N_PERM, seed=SEED)
    n_sig_ch     = int(sig_mask[roi_indices].sum())
    roi_results[roi_name] = dict(
        mean=mean, ci_lo=lo, ci_hi=hi,
        p_raw=p_val, n_sig=n_sig_ch, n_ch=len(roi_chs)
    )

roi_p_arr = np.array([roi_results[r]['p_raw'] for r in roi_names])
roi_adj_p, roi_sig = fdr_bh(roi_p_arr, alpha=FDR_ALPHA)

print(f'ROI-level importance (FDR alpha={FDR_ALPHA}):')
print(f'{"ROI":>16}  {"Mean":>9}  {"95% CI":>22}  {"p_raw":>8}  {"q_adj":>8}  {"Sig?":>5}  {"Ch_sig":>6}')
print('-' * 88)
for i, roi_name in enumerate(roi_names):
    r = roi_results[roi_name]
    sig_str = 'YES' if roi_sig[i] else 'no'
    print(f'{roi_name:>16}  {r["mean"]:>9.5f}  [{r["ci_lo"]:+.5f}, {r["ci_hi"]:+.5f}]  '
          f'{r["p_raw"]:>8.4f}  {roi_adj_p[i]:>8.4f}  {sig_str:>5}  '
          f'{r["n_sig"]:>2}/{r["n_ch"]:<2}')

## Section G — Visualization

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# ── Left: per-channel importance ─────────────────────────────────────────────
ax = axes[0]
colors = ['#1565C0' if sig_mask[i] else '#B0BEC5' for i in range(N_CHANNELS)]
bars = ax.bar(range(N_CHANNELS), imp_ch, color=colors, alpha=0.85, width=0.8)

# Add bootstrap CI error bars for significant channels only
for i in sig_indices:
    ax.errorbar(i, ci_arr[i, 0],
                yerr=[[ci_arr[i, 0]-ci_arr[i, 1]], [ci_arr[i, 2]-ci_arr[i, 0]]],
                fmt='none', color='#0D47A1', capsize=2, linewidth=0.8)

ax.axhline(0, color='black', linewidth=0.7, linestyle='--')
ax.set_xlabel('Channel index', fontsize=11)
ax.set_ylabel('Importance  (Δ accuracy)', fontsize=11)
ax.set_title(f'Per-channel LOCO importance  (blue = FDR q<{FDR_ALPHA})', fontsize=11)
ax.set_xlim(-1, N_CHANNELS)

# Annotate top significant channel names
top_sig = sig_indices[np.argsort(imp_ch[sig_indices])[::-1]][:8]
for idx in top_sig:
    ax.text(idx, imp_ch[idx] + 0.002, CHANNEL_NAMES[idx],
            ha='center', va='bottom', fontsize=6, rotation=75)

# ── Right: ROI-level importance ──────────────────────────────────────────────
ax2 = axes[1]
roi_means  = [roi_results[r]['mean']  for r in roi_names]
roi_ci_lo  = [roi_results[r]['ci_lo'] for r in roi_names]
roi_ci_hi  = [roi_results[r]['ci_hi'] for r in roi_names]
roi_colors = ['#B71C1C' if roi_sig[i] else '#78909C' for i in range(len(roi_names))]
yerr = [
    [roi_means[i] - roi_ci_lo[i] for i in range(len(roi_names))],
    [roi_ci_hi[i] - roi_means[i] for i in range(len(roi_names))],
]
x = np.arange(len(roi_names))
ax2.bar(x, roi_means, color=roi_colors, alpha=0.85, yerr=yerr, capsize=5)
ax2.axhline(0, color='black', linewidth=0.7, linestyle='--')
ax2.set_xticks(x)
ax2.set_xticklabels(roi_names, rotation=30, ha='right', fontsize=10)
ax2.set_ylabel('Importance  (Δ accuracy)', fontsize=11)
ax2.set_title(f'ROI-level importance  (red = FDR q<{FDR_ALPHA})', fontsize=11)

plt.tight_layout()
out_png = OUTPUT_DIR / 'aadnet_fdr_channel_importance.png'
out_pdf = OUTPUT_DIR / 'aadnet_fdr_channel_importance.pdf'
plt.savefig(out_png, dpi=150, bbox_inches='tight')
plt.savefig(out_pdf,           bbox_inches='tight')
plt.show()
print(f'Saved:\n  {out_png}\n  {out_pdf}')

## Section H — Per-Window FDR Analysis

Run the same FDR pipeline separately for each decision-window length to see
whether channel importance is consistent across window sizes.

In [ ]:
per_window = {}   # {win_sec: {'sig_channels': array, 'sig_rois': array}}

print(f'{"Window":>8}  {"Sig channels":>13}  {"Sig ROIs":>9}')
print('-' * 36)

for w, win_sec in enumerate(WINDOWS):
    imp_w = importance[w]   # (18, 64)  subjects x channels

    # Channel-level FDR
    p_w = np.array([
        sign_flip_pvalue(imp_w[:, c], n_perm=2000, seed=SEED)
        for c in range(N_CHANNELS)
    ])
    _, sig_ch_w = fdr_bh(p_w, alpha=FDR_ALPHA)

    # ROI-level FDR
    p_roi_w = []
    for roi_name, roi_chs in ROI_MAP.items():
        roi_idx = [CHANNEL_NAMES.index(ch) for ch in roi_chs]
        roi_imp_w = imp_w[:, roi_idx].mean(axis=1)
        p_roi_w.append(sign_flip_pvalue(roi_imp_w, n_perm=2000, seed=SEED))
    _, sig_roi_w = fdr_bh(np.array(p_roi_w), alpha=FDR_ALPHA)

    per_window[win_sec] = {'sig_channels': sig_ch_w, 'sig_rois': sig_roi_w}
    print(f'{win_sec:>6}s    {sig_ch_w.sum():>10}     {sig_roi_w.sum():>6}')

In [ ]:
# Heatmap: which channels are significant at each window?
sig_matrix = np.zeros((N_WINDOWS, N_CHANNELS), dtype=int)
for w, win_sec in enumerate(WINDOWS):
    sig_matrix[w] = per_window[win_sec]['sig_channels'].astype(int)

consistency = sig_matrix.sum(axis=0)   # how many windows each channel is sig
consistent_chs = np.where(consistency >= 4)[0]

print(f'Channels significant in >= 4 of 6 windows: {len(consistent_chs)}')
if len(consistent_chs) > 0:
    print(f'{"Ch":>4}  {"Name":>8}  {"ROI":>16}  {"#Windows sig":>13}')
    print('-' * 48)
    for idx in consistent_chs[np.argsort(consistency[consistent_chs])[::-1]]:
        ch = CHANNEL_NAMES[idx]
        print(f'{idx:>4}  {ch:>8}  {CH_TO_ROI[ch]:>16}  {consistency[idx]:>13}')

# Plot consistency heatmap
fig, ax = plt.subplots(figsize=(18, 3))
im = ax.imshow(sig_matrix, aspect='auto', cmap='Blues', vmin=0, vmax=1,
               interpolation='none')
ax.set_yticks(range(N_WINDOWS))
ax.set_yticklabels([f'{w}s' for w in WINDOWS])
ax.set_xlabel('Channel index')
ax.set_ylabel('Window size')
ax.set_title(f'Channels significant (FDR q<{FDR_ALPHA}) per window  (dark = significant)')

# Mark channels that are in ROIs
boundary = 0
for roi_name, roi_chs in ROI_MAP.items():
    boundary += len(roi_chs)
    ax.axvline(boundary - 0.5, color='red', linewidth=0.5, alpha=0.4)

plt.tight_layout()
out_hm = OUTPUT_DIR / 'aadnet_fdr_window_heatmap.png'
plt.savefig(out_hm, dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved: {out_hm}')

## Section I — Save Summary

In [ ]:
def _float(x):
    return float(x) if isinstance(x, (np.floating, np.integer)) else x

summary = {
    'parameters': {
        'n_subjects': N_SUBJECTS,
        'n_channels': N_CHANNELS,
        'n_windows': N_WINDOWS,
        'windows_sec': WINDOWS,
        'fdr_alpha': FDR_ALPHA,
        'n_boot': N_BOOT,
        'n_perm': N_PERM,
    },
    'channel_results': {
        CHANNEL_NAMES[i]: {
            'roi': CH_TO_ROI[CHANNEL_NAMES[i]],
            'importance_mean': _float(ci_arr[i, 0]),
            'ci_lo': _float(ci_arr[i, 1]),
            'ci_hi': _float(ci_arr[i, 2]),
            'p_raw': _float(p_values[i]),
            'q_adj': _float(adj_p[i]),
            'significant': bool(sig_mask[i]),
        }
        for i in range(N_CHANNELS)
    },
    'roi_results': {
        roi_names[i]: {
            **{k: _float(v) for k, v in roi_results[roi_names[i]].items()},
            'q_adj': _float(roi_adj_p[i]),
            'significant': bool(roi_sig[i]),
        }
        for i in range(len(roi_names))
    },
    'significant_channels': [CHANNEL_NAMES[i] for i in sig_sorted],
    'consistent_channels_ge4windows': [CHANNEL_NAMES[i] for i in consistent_chs.tolist()],
}

out_json = OUTPUT_DIR / 'aadnet_fdr_summary.json'
with open(out_json, 'w', encoding='utf-8') as f:
    json.dump(summary, f, indent=2)
print(f'Summary saved: {out_json}')
print(f'\nSig channels : {summary["significant_channels"]}')
print(f'Consistent   : {summary["consistent_channels_ge4windows"]}')